## Apply optimizations to ONNX model

Now that we have an ONNX model, we can apply some basic optimizations. After completing this section, you should be able to apply:

-   graph optimizations, e.g. fusing operations
-   post-training quantization (dynamic and static)
-   and hardware-specific execution providers

to improve inference performance.

You will execute this notebook *in a Jupyter container running on a compute instance*, not on the general-purpose Chameleon Jupyter environment from which you provision resources.

Since we are going to evaluate several models, we’ll define a benchmark function here to help us compare them:

In [3]:
!pip install "optimum[onnxruntime]" transformers huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 117.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.1/801.1 kB 96.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 135.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 132.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 70.0 MB/s eta 0:00:00


In [4]:
!pip install pytorch-ignite

In [5]:
# runs in jupyter container on node-serve-model
import os
import time
import numpy as np
import torch
import onnx
import onnxruntime as ort
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [6]:
# runs in jupyter container on node-serve-model
# Prepare test dataset
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader,Subset
from transformers import AutoTokenizer

# Load your eval file
df = pd.read_json("/mnt/eval.jsonl", lines=True)

model_id = "google-t5/t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_id)

max_input_len = 512
max_target_len = 256


class RecipeEvalDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_input_len=512, max_target_len=256):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Keep the "bad" input mostly intact
        input_text = row["input"].strip()
        target_text = row["target"].strip()

        # Optional: remove metadata noise only
        input_text = input_text.replace("<source:web_scrape>", "").strip()

        model_inputs = self.tokenizer(
            input_text,
            max_length=self.max_input_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        labels = self.tokenizer(
            text_target=target_text,
            max_length=self.max_target_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        input_ids = model_inputs["input_ids"].squeeze(0)
        attention_mask = model_inputs["attention_mask"].squeeze(0)
        labels = labels["input_ids"].squeeze(0)

        # Optional: ignore padding tokens in loss
        labels[labels == tokenizer.pad_token_id] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "raw_input": input_text,
            "raw_target": target_text,
        }


dataset = RecipeEvalDataset(df, tokenizer, max_input_len, max_target_len)

dataloader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=False,
)

# from torch.utils.data import Subset, DataLoader

subset_size = 200
small_dataset = Subset(dataset, range(subset_size))

small_dataloader = DataLoader(
    small_dataset,
    batch_size=4,
    shuffle=False,
)

batch = next(iter(small_dataloader))
print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["labels"].shape)
print(batch["raw_input"][0][:200])
print(batch["raw_target"][0][:200])

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

torch.Size([4, 512])
torch.Size([4, 512])
torch.Size([4, 256])
fix recipe: Title: Valentine's Thins
Ingredients: 36 RITZ Crackers with Whole Wheat | 2 pkg. (4 oz. each) BAKER'S Semi-Sweet Chocolate, broken into pieces, melted | 36 berry-shaped gummy candies (abou
Title: Valentine's Thins
Ingredients: 36 Ritz Crackers with Whole Wheat | 1 pkg. (225 g) Baker's Semi-Sweet Chocolate, melted | 36 Maynards Swedish Berries (about 2 pkg. [64 g each]) | 2 oz. Baker's W


In [3]:
# runs in jupyter container on node-serve-model
def benchmark_session(ort_session):

    print(f"Execution provider: {ort_session.get_providers()}")

    ## Benchmark accuracy

    correct = 0
    total = 0
    for images, labels in test_loader:
        images_np = images.numpy()
        outputs = ort_session.run(None, {ort_session.get_inputs()[0].name: images_np})[0]
        predicted = np.argmax(outputs, axis=1)
        total += labels.size(0)
        correct += (predicted == labels.numpy()).sum()
    accuracy = (correct / total) * 100

    print(f"Accuracy: {accuracy:.2f}% ({correct}/{total} correct)")

    ## Benchmark inference latency for single sample

    num_trials = 100  # Number of trials

    # Get a single sample from the test data

    single_sample, _ = next(iter(test_loader))  
    single_sample = single_sample[:1].numpy()

    # Warm-up run
    ort_session.run(None, {ort_session.get_inputs()[0].name: single_sample})

    latencies = []
    for _ in range(num_trials):
        start_time = time.time()
        ort_session.run(None, {ort_session.get_inputs()[0].name: single_sample})
        latencies.append(time.time() - start_time)

    print(f"Inference Latency (single sample, median): {np.percentile(latencies, 50) * 1000:.2f} ms")
    print(f"Inference Latency (single sample, 95th percentile): {np.percentile(latencies, 95) * 1000:.2f} ms")
    print(f"Inference Latency (single sample, 99th percentile): {np.percentile(latencies, 99) * 1000:.2f} ms")
    print(f"Inference Throughput (single sample): {num_trials/np.sum(latencies):.2f} FPS")

    ## Benchmark batch throughput

    num_batches = 50  # Number of trials

    # Get a batch from the test data
    batch_input, _ = next(iter(test_loader))  
    batch_input = batch_input.numpy()

    # Warm-up run
    ort_session.run(None, {ort_session.get_inputs()[0].name: batch_input})

    batch_times = []
    for _ in range(num_batches):
        start_time = time.time()
        ort_session.run(None, {ort_session.get_inputs()[0].name: batch_input})
        batch_times.append(time.time() - start_time)

    batch_fps = (batch_input.shape[0] * num_batches) / np.sum(batch_times) 
    print(f"Batch Throughput: {batch_fps:.2f} FPS")


In [7]:
import time
import numpy as np
from tqdm.auto import tqdm
from ignite.metrics import RougeL


def benchmark_onnx_model(onnx_model, dataloader, tokenizer, max_new_tokens=256):
    print("ONNX model loaded with Hugging Face Optimum")

    # ----------------------------
    # 1. ROUGE-L
    # ----------------------------
    predictions = []
    references = []

    rouge_l = RougeL(multiref="best")
    rouge_l.reset()

    for batch in tqdm(dataloader, desc="Evaluating ONNX model"):
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        refs = batch["raw_target"]

        output_ids = onnx_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
        )

        preds = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

        predictions.extend(preds)
        references.extend(refs)

        candidates = [p.split() for p in preds]
        gold_refs = [[r.split()] for r in refs]
        rouge_l.update((candidates, gold_refs))

    rouge_scores = rouge_l.compute()

    print(f"Rouge-L-P: {rouge_scores['Rouge-L-P']:.4f}")
    print(f"Rouge-L-R: {rouge_scores['Rouge-L-R']:.4f}")
    print(f"Rouge-L-F: {rouge_scores['Rouge-L-F']:.4f}")

    # ----------------------------
    # 2. Single-sample latency
    # ----------------------------
    num_trials = 100

    batch = next(iter(dataloader))
    single_input_ids = batch["input_ids"][0].unsqueeze(0)
    single_attention_mask = batch["attention_mask"][0].unsqueeze(0)

    # Warm-up
    _ = onnx_model.generate(
        input_ids=single_input_ids,
        attention_mask=single_attention_mask,
        max_new_tokens=max_new_tokens,
    )

    latencies = []
    for _ in tqdm(range(num_trials), desc="Single-sample latency"):
        start_time = time.perf_counter()
        _ = onnx_model.generate(
            input_ids=single_input_ids,
            attention_mask=single_attention_mask,
            max_new_tokens=max_new_tokens,
        )
        latencies.append(time.perf_counter() - start_time)

    print(f"Inference Latency (single sample, median): {np.percentile(latencies, 50) * 1000:.2f} ms")
    print(f"Inference Latency (single sample, 95th percentile): {np.percentile(latencies, 95) * 1000:.2f} ms")
    print(f"Inference Latency (single sample, 99th percentile): {np.percentile(latencies, 99) * 1000:.2f} ms")
    print(f"Inference Throughput (single sample): {num_trials / np.sum(latencies):.2f} samples/sec")

    # ----------------------------
    # 3. Batch throughput
    # ----------------------------
    num_batches = 50

    batch = next(iter(dataloader))
    batch_input_ids = batch["input_ids"]
    batch_attention_mask = batch["attention_mask"]

    # Warm-up
    _ = onnx_model.generate(
        input_ids=batch_input_ids,
        attention_mask=batch_attention_mask,
        max_new_tokens=max_new_tokens,
    )

    batch_times = []
    for _ in tqdm(range(num_batches), desc="Batch throughput"):
        start_time = time.perf_counter()
        _ = onnx_model.generate(
            input_ids=batch_input_ids,
            attention_mask=batch_attention_mask,
            max_new_tokens=max_new_tokens,
        )
        batch_times.append(time.perf_counter() - start_time)

    batch_fps = (batch_input_ids.shape[0] * num_batches) / np.sum(batch_times)
    print(f"Batch Throughput: {batch_fps:.2f} samples/sec")

/opt/conda/lib/python3.12/site-packages/ignite/handlers/checkpoint.py:16: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer


### Apply basic graph optimizations

Let’s start by applying some basic [graph optimizations](https://onnxruntime.ai/docs/performance/model-optimizations/graph-optimizations.html#onlineoffline-mode), e.g. fusing operations.

We will save the model after applying graph optimizations to `models/food11_optimized.onnx`, then evaluate that model in a new session.

In [8]:
# runs in jupyter container on node-serve-model
# onnx_model_path = "models/food11.onnx"
# optimized_model_path = "models/food11_optimized.onnx"

# session_options = ort.SessionOptions()
# session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED # apply graph optimizations
# # session_options.optimized_model_filepath = "model/optimized_onnx" 

# ort_session = ort.InferenceSession(onnx_model_path, sess_options=session_options, providers=['CPUExecutionProvider'])



from pathlib import Path

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from optimum.onnxruntime import ORTModelForSeq2SeqLM
import onnxruntime as ort

model_id = "google-t5/t5-small"



tokenizer = AutoTokenizer.from_pretrained(model_id)

# onnx_model = ORTModelForSeq2SeqLM.from_pretrained(
#     model_id,
#     export=True,
#     providers=["CPUExecutionProvider"],
#     session_options=session_options,
# )

base_dir = Path("/home/jovyan/work/models")
# pt_dir = base_dir / "pytorch"
onnx_dir = base_dir / "onnx/onnx_optimized"

# pt_dir.mkdir(parents=True, exist_ok=True)
onnx_dir.mkdir(parents=True, exist_ok=True)

# tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# # save PyTorch version
# pt_model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
# # pt_model.save_pretrained(pt_dir)
# # tokenizer.save_pretrained(pt_dir)

# export and save ONNX version
session_options = ort.SessionOptions()
session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED # apply graph optimizations
onnx_model = ORTModelForSeq2SeqLM.from_pretrained(
    model_id, 
    export=True,providers=["CPUExecutionProvider"],
    session_options=session_options,)
onnx_model.save_pretrained(onnx_dir)
# tokenizer.save_pretrained(onnx_dir)

# print("Saved PyTorch model to:", pt_dir)
print("Saved ONNX model to:", onnx_dir)

The model google-t5/t5-small was already converted to ONNX but got `export=True`, the model will be converted to ONNX once again. Don't forget to save the resulting model with `.save_pretrained()`


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

/opt/conda/lib/python3.12/site-packages/transformers/models/t5/modeling_t5.py:1271: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if sequence_length != 1:
/opt/conda/lib/python3.12/site-packages/transformers/cache_utils.py:132: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if not self.is_initialized or self.keys.numel() == 0:


Saved ONNX model to: /home/jovyan/work/models/onnx/onnx_optimized


Download the `food11_optimized.onnx` model from inside the `models` directory.

To see the effect of the graph optimizations, we can visualize the models using [Netron](https://netron.app/). Upload the original `food11.onnx` and review the graph. Then, upload the `food11_optimized.onnx` and see what has changed in the “optimized” graph.

Next, evaluate the optimized model. The graph optimizations may improve the inference performance, may have negligible effect, OR they can make it worse, depending on the model and the hardware environment in which the model is executed.

In [10]:
# runs in jupyter container on node-serve-model
benchmark_onnx_model(
    onnx_model=onnx_model,
    dataloader=small_dataloader,
    tokenizer=tokenizer,
    max_new_tokens=256,
)

ONNX model loaded with Hugging Face Optimum


Evaluating ONNX model:   0%|          | 0/50 [00:00<?, ?it/s]

Rouge-L-P: 0.4636
Rouge-L-R: 0.2368
Rouge-L-F: 0.2368


Single-sample latency:   0%|          | 0/100 [00:00<?, ?it/s]

Inference Latency (single sample, median): 695.01 ms
Inference Latency (single sample, 95th percentile): 731.16 ms
Inference Latency (single sample, 99th percentile): 736.59 ms
Inference Throughput (single sample): 1.43 samples/sec


Batch throughput:   0%|          | 0/50 [00:00<?, ?it/s]

Batch Throughput: 3.61 samples/sec


<!--

On gigaio AMD EPYC:


Execution provider: ['CPUExecutionProvider']
Accuracy: 90.59% (3032/3347 correct)
Inference Latency (single sample, median): 8.70 ms
Inference Latency (single sample, 95th percentile): 8.88 ms
Inference Latency (single sample, 99th percentile): 9.24 ms
Inference Throughput (single sample): 114.63 FPS
Batch Throughput: 1153.63 FPS

On liqid Intel:

Execution provider: ['CPUExecutionProvider']
Accuracy: 90.59% (3032/3347 correct)
Inference Latency (single sample, median): 4.63 ms
Inference Latency (single sample, 95th percentile): 4.67 ms
Inference Latency (single sample, 99th percentile): 4.75 ms
Inference Throughput (single sample): 214.45 FPS
Batch Throughput: 2488.54 FPS

-->

### Apply post training quantization

We will continue our quest to improve inference speed! The next optimization we will attempt is quantization.

There are many frameworks that offer quantization - for our Food11 model, we could:

-   use [PyTorch quantization](https://docs.pytorch.org/ao/stable/index.html)
-   use [ONNX quantization](https://onnxruntime.ai/docs/performance/model-optimizations/quantization.html)
-   use [Intel Neural Compressor](https://intel.github.io/neural-compressor/latest/index.html) (which supports PyTorch and ONNX models)
-   use [NNCF](https://github.com/openvinotoolkit/nncf) if we plan to use the OpenVINO execution provider
-   etc…

These frameworks vary in the type of quantization they support, the range of operations that may be quantized, and many other details.

We will use Intel Neural Compressor, which in addition to supporting many ML frameworks and many types of quantization has an interesting feature: it supports quantization up to a specified evaluation threshold. In other words, we can specify “quantize as much as possible, but without losing more than 0.01 accuracy” and Intel Neural Compressor will find the best quantized version of the model that does not lose more than 0.01 accuracy.

Post-training quantization comes in two main types. In both types, FP32 values will be converted in INT8, using

$$\texttt{val}\_\texttt{quant} = \texttt{round}\left(\frac{\texttt{val}\_\texttt{fp32}}{\texttt{scale}}\right) + \texttt{zero}\_\texttt{point}$$

but they differ with respect to when and how the quantization parameters “scale” and “zero point” are computed:

-   dynamic quantization: weights are quantized in advance and stored in INT8 representation. The quantization parameters for the activations are computed during inference.
-   static quantization: weights are quantized in advance and stored in INT8, and the quantization parameters are also set in advance for activations. This approach requires the use of a “calibration dataset” during quantization, to set the quantization parameters for the activations.

#### Dynamic quantization with onnxruntime

We will start with dynamic quantization. No calibration dataset is required.

In [18]:
import sys
!{sys.executable} -m pip install --no-cache-dir -U ml_dtypes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 79.8 MB/s eta 0:00:00


In [19]:
# from onnxruntime.quantization import quantize_dynamic, QuantType

# quantize_dynamic(
#     model_input="/home/jovyan/work/models/onnx/model.onnx",
#     model_output="/home/jovyan/work/models/onnx-int8-dynamic/model.onnx",
#     weight_type=QuantType.QInt8,
# )


# from optimum.onnxruntime import ORTModelForSeq2SeqLM
# from transformers import AutoTokenizer

# model_dir = "/home/jovyan/work/models/onnx"

# tokenizer = AutoTokenizer.from_pretrained(model_dir)
# onnx_model = ORTModelForSeq2SeqLM.from_pretrained(model_dir)

from pathlib import Path
from onnxruntime.quantization import quantize_dynamic, QuantType

src_dir = Path("/home/jovyan/work/models/onnx/onnx")
dst_dir = Path("/home/jovyan/work/models/onnx/onnx-int8-dynamic")
dst_dir.mkdir(parents=True, exist_ok=True)

onnx_files = [
    "encoder_model.onnx",
    "decoder_model.onnx",
    "decoder_with_past_model.onnx",
]

for name in onnx_files:
    src = src_dir / name
    dst = dst_dir / name

    quantize_dynamic(
        model_input=str(src),
        model_output=str(dst),
        weight_type=QuantType.QInt8,
    )

    print(f"Quantized {src.name} -> {dst}")

Quantized encoder_model.onnx -> /home/jovyan/work/models/onnx/onnx-int8-dynamic/encoder_model.onnx


Quantized decoder_model.onnx -> /home/jovyan/work/models/onnx/onnx-int8-dynamic/decoder_model.onnx


Quantized decoder_with_past_model.onnx -> /home/jovyan/work/models/onnx/onnx-int8-dynamic/decoder_with_past_model.onnx


In [20]:
import shutil
from pathlib import Path

src_dir = Path("/home/jovyan/work/models/onnx/onnx")
dst_dir = Path("/home/jovyan/work/models/onnx/onnx-int8-dynamic")

for name in [
    "config.json",
    "generation_config.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "spiece.model",
    "tokenizer.json",
]:
    src = src_dir / name
    if src.exists():
        shutil.copy2(src, dst_dir / name)
        print(f"Copied {name}")

Copied config.json
Copied generation_config.json
Copied tokenizer_config.json
Copied special_tokens_map.json
Copied tokenizer.json


In [23]:
# import onnxruntime as ort
# from optimum.onnxruntime import ORTModelForSeq2SeqLM
# from transformers import AutoTokenizer

# quant_dir = "/home/jovyan/work/models/onnx/onnx-int8-dynamic"

# session_options = ort.SessionOptions()
# session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

# tokenizer = AutoTokenizer.from_pretrained(quant_dir)

# onnx_model_dynamic = ORTModelForSeq2SeqLM.from_pretrained(
#     quant_dir,
#     providers=["CPUExecutionProvider"],
#     session_options=session_options,
# )

import onnxruntime as ort
from optimum.onnxruntime import ORTModelForSeq2SeqLM

quant_dir = "/home/jovyan/work/models/onnx/onnx-int8-dynamic"

session_options = ort.SessionOptions()
session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

onnx_model_dynamic = ORTModelForSeq2SeqLM.from_pretrained(
    quant_dir,
    encoder_file_name="encoder_model.onnx",
    decoder_file_name="decoder_model.onnx",
    decoder_with_past_file_name="decoder_with_past_model.onnx",
    providers=["CPUExecutionProvider"],
    session_options=session_options,
)

In [26]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(quant_dir)

inputs = tokenizer("fix recipe: huh", return_tensors="pt")
out = onnx_model_dynamic.generate(**inputs, max_new_tokens=16)
print(tokenizer.decode(out[0], skip_special_tokens=True))

fix recipe recipe recipe recipe recipe recipe: huh


In [27]:
benchmark_onnx_model(
    onnx_model=onnx_model_dynamic,
    dataloader=small_dataloader,
    tokenizer=tokenizer,
    max_new_tokens=256,
)

ONNX model loaded with Hugging Face Optimum


Evaluating ONNX model:   0%|          | 0/50 [00:00<?, ?it/s]

Rouge-L-P: 0.4728
Rouge-L-R: 0.2418
Rouge-L-F: 0.2418


Single-sample latency:   0%|          | 0/100 [00:00<?, ?it/s]

Inference Latency (single sample, median): 460.93 ms
Inference Latency (single sample, 95th percentile): 506.03 ms
Inference Latency (single sample, 99th percentile): 527.95 ms
Inference Throughput (single sample): 2.14 samples/sec


Batch throughput:   0%|          | 0/50 [00:00<?, ?it/s]

Batch Throughput: 4.02 samples/sec


In [29]:
# runs in jupyter container on node-serve-model
model_path_1 = os.path.join(quant_dir, "decoder_model.onnx")
model_path_2 = os.path.join(quant_dir, "decoder_with_past_model.onnx")
model_path_3 = os.path.join(quant_dir, "encoder_model.onnx")
model_size_1= os.path.getsize(model_path_1)
model_size_2= os.path.getsize(model_path_2)
model_size_3= os.path.getsize(model_path_3)
onnx_dyanmic_quantization_model_size = model_size_1+model_size_2+model_size_3
print(f"ONNX Model Size on Disk: {onnx_dyanmic_quantization_model_size/ (1e6) :.2f} MB")

ONNX Model Size on Disk: 149.24 MB


#### Dynamic quantization with neural_compressor

In [30]:
# runs in jupyter container on node-serve-model
import neural_compressor
from neural_compressor import quantization

In [31]:
# runs in jupyter container on node-serve-model
# Load ONNX model into Intel Neural Compressor
from pathlib import Path
import shutil

from neural_compressor.config import PostTrainingQuantConfig
from neural_compressor import quantization
import neural_compressor.model.onnx_model

src_dir = Path("/home/jovyan/work/models/onnx/onnx")
dst_dir = Path("/home/jovyan/work/models/onnx/onnx-inc-dynamic")
dst_dir.mkdir(parents=True, exist_ok=True)

config_ptq = PostTrainingQuantConfig(approach="dynamic")

onnx_files = [
    "encoder_model.onnx",
    "decoder_model.onnx",
    "decoder_with_past_model.onnx",
]

for name in onnx_files:
    model_path = src_dir / name
    fp32_model = neural_compressor.model.onnx_model.ONNXModel(str(model_path))

    q_model = quantization.fit(
        model=fp32_model,
        conf=config_ptq,
    )

    out_path = dst_dir / name
    q_model.save_model_to_file(str(out_path))
    print(f"Saved quantized file: {out_path}")

2026-04-06 18:33:25 [WARNING] Provided eval_func is not a plain function; security checks limited.
2026-04-06 18:33:25 [INFO] Start auto tuning.
2026-04-06 18:33:25 [INFO] Quantize model without tuning!
2026-04-06 18:33:25 [INFO] Quantize the model with default configuration without evaluating the model.                To perform the tuning process, please either provide an eval_func or provide an                    eval_dataloader an eval_metric.
2026-04-06 18:33:25 [INFO] Adaptor has 5 recipes.
2026-04-06 18:33:25 [INFO] 0 recipes specified by user.
2026-04-06 18:33:25 [INFO] 3 recipes require future tuning.
2026-04-06 18:33:26 [INFO] *** Initialize auto tuning
2026-04-06 18:33:26 [INFO] {
2026-04-06 18:33:26 [INFO]     'PostTrainingQuantConfig': {
2026-04-06 18:33:26 [INFO]         'AccuracyCriterion': {
2026-04-06 18:33:26 [INFO]             'criterion': 'relative',
2026-04-06 18:33:26 [INFO]             'higher_is_better': True,
2026-04-06 18:33:26 [INFO]             'tolerable_lo

Saved quantized file: /home/jovyan/work/models/onnx/onnx-inc-dynamic/encoder_model.onnx


2026-04-06 18:33:32 [WARNING] Provided eval_func is not a plain function; security checks limited.
2026-04-06 18:33:32 [INFO] Start auto tuning.
2026-04-06 18:33:32 [INFO] Quantize model without tuning!
2026-04-06 18:33:32 [INFO] Quantize the model with default configuration without evaluating the model.                To perform the tuning process, please either provide an eval_func or provide an                    eval_dataloader an eval_metric.
2026-04-06 18:33:32 [INFO] Adaptor has 5 recipes.
2026-04-06 18:33:32 [INFO] 0 recipes specified by user.
2026-04-06 18:33:32 [INFO] 3 recipes require future tuning.
2026-04-06 18:33:32 [INFO] *** Initialize auto tuning
2026-04-06 18:33:32 [INFO] {
2026-04-06 18:33:32 [INFO]     'PostTrainingQuantConfig': {
2026-04-06 18:33:32 [INFO]         'AccuracyCriterion': {
2026-04-06 18:33:32 [INFO]             'criterion': 'relative',
2026-04-06 18:33:32 [INFO]             'higher_is_better': True,
2026-04-06 18:33:32 [INFO]             'tolerable_lo

Saved quantized file: /home/jovyan/work/models/onnx/onnx-inc-dynamic/decoder_model.onnx


2026-04-06 18:33:43 [WARNING] Provided eval_func is not a plain function; security checks limited.
2026-04-06 18:33:43 [INFO] Start auto tuning.
2026-04-06 18:33:43 [INFO] Quantize model without tuning!
2026-04-06 18:33:43 [INFO] Quantize the model with default configuration without evaluating the model.                To perform the tuning process, please either provide an eval_func or provide an                    eval_dataloader an eval_metric.
2026-04-06 18:33:43 [INFO] Adaptor has 5 recipes.
2026-04-06 18:33:43 [INFO] 0 recipes specified by user.
2026-04-06 18:33:43 [INFO] 3 recipes require future tuning.
2026-04-06 18:33:43 [INFO] *** Initialize auto tuning
2026-04-06 18:33:43 [INFO] {
2026-04-06 18:33:43 [INFO]     'PostTrainingQuantConfig': {
2026-04-06 18:33:43 [INFO]         'AccuracyCriterion': {
2026-04-06 18:33:43 [INFO]             'criterion': 'relative',
2026-04-06 18:33:43 [INFO]             'higher_is_better': True,
2026-04-06 18:33:43 [INFO]             'tolerable_lo

Saved quantized file: /home/jovyan/work/models/onnx/onnx-inc-dynamic/decoder_with_past_model.onnx


In [32]:
for name in [
    "config.json",
    "generation_config.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "spiece.model",
    "tokenizer.json",
]:
    src = src_dir / name
    if src.exists():
        shutil.copy2(src, dst_dir / name)

In [34]:
import onnxruntime as ort
from optimum.onnxruntime import ORTModelForSeq2SeqLM
from transformers import AutoTokenizer

session_options = ort.SessionOptions()
session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

tokenizer = AutoTokenizer.from_pretrained(dst_dir)

onnx_model_inc = ORTModelForSeq2SeqLM.from_pretrained(
    dst_dir,
    encoder_file_name="encoder_model.onnx",
    decoder_file_name="decoder_model.onnx",
    decoder_with_past_file_name="decoder_with_past_model.onnx",
    providers=["CPUExecutionProvider"],
    session_options=session_options,
)

In [35]:
quant_dir = "models/onnx/onnx-inc-dynamic"
model_path_1 = os.path.join(quant_dir, "decoder_model.onnx")
model_path_2 = os.path.join(quant_dir, "decoder_with_past_model.onnx")
model_path_3 = os.path.join(quant_dir, "encoder_model.onnx")
model_size_1= os.path.getsize(model_path_1)
model_size_2= os.path.getsize(model_path_2)
model_size_3= os.path.getsize(model_path_3)
onnx_dyanmic_quantization_inc_model_size = model_size_1+model_size_2+model_size_3
print(f"ONNX Model Size on Disk: {onnx_dyanmic_quantization_inc_model_size/ (1e6) :.2f} MB")

ONNX Model Size on Disk: 149.69 MB


In [36]:
benchmark_onnx_model(
    onnx_model=onnx_model_inc,
    dataloader=small_dataloader,
    tokenizer=tokenizer,
    max_new_tokens=256,
)

ONNX model loaded with Hugging Face Optimum


Evaluating ONNX model:   0%|          | 0/50 [00:00<?, ?it/s]

Rouge-L-P: 0.4973
Rouge-L-R: 0.2531
Rouge-L-F: 0.2531


Single-sample latency:   0%|          | 0/100 [00:00<?, ?it/s]

Inference Latency (single sample, median): 507.16 ms
Inference Latency (single sample, 95th percentile): 548.98 ms
Inference Latency (single sample, 99th percentile): 575.39 ms
Inference Throughput (single sample): 1.95 samples/sec


Batch throughput:   0%|          | 0/50 [00:00<?, ?it/s]

Batch Throughput: 3.70 samples/sec


Next, evaluate the quantized model. Since we are saving weights in integer form, the model size is smaller. With respect to inference time, however, while the integer operations may be faster than their FP32 equivalents, the dynamic quantization and dequantization of activations may add more compute time than we save from integer operations.

<!-- 

On liqid AMD EPYC

Model Size on Disk: 2.42 MB
Execution provider: ['CPUExecutionProvider']
Accuracy: 82.04% (2746/3347 correct)
Inference Latency (single sample, median): 22.32 ms
Inference Latency (single sample, 95th percentile): 22.97 ms
Inference Latency (single sample, 99th percentile): 23.14 ms
Inference Throughput (single sample): 44.71 FPS
Batch Throughput: 38.34 FPS

On liqid Intel

Execution provider: ['CPUExecutionProvider']
Accuracy: 84.58% (2831/3347 correct)
Inference Latency (single sample, median): 28.29 ms
Inference Latency (single sample, 95th percentile): 29.00 ms
Inference Latency (single sample, 99th percentile): 29.07 ms
Inference Throughput (single sample): 35.28 FPS

-->

#### Static quantization

Next, we will try static quantization with a calibration dataset.

First, let’s prepare the calibration dataset. This dataset will also be used to evaluate the quantized model, to see if it meets the accuracy criterion we will set.

In [37]:
# runs in jupyter container on node-serve-model
import neural_compressor
from neural_compressor import quantization
from torchvision import datasets, transforms

In [ ]:
# runs in jupyter container on node-serve-model
food_11_data_dir = os.getenv("FOOD11_DATA_DIR", "Food-11")
val_test_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset
val_dataset = datasets.ImageFolder(root=os.path.join(food_11_data_dir, 'validation'), transform=val_test_transform)
eval_dataloader = neural_compressor.data.DataLoader(framework='onnxruntime', dataset=val_dataset)

Then, we’ll configure the quantizer. We’ll start with a more aggressive quantization strategy - we will prefer to quantize as much as possible, as long as the accuracy of the quantized model is not more than **0.05** less than the accuracy of the original FP32 model.

In [ ]:
# runs in jupyter container on node-serve-model
# Load ONNX model into Intel Neural Compressor
model_path = "models/food11.onnx"
fp32_model = neural_compressor.model.onnx_model.ONNXModel(model_path)

# Configure the quantizer
config_ptq = neural_compressor.PostTrainingQuantConfig(
    accuracy_criterion = neural_compressor.config.AccuracyCriterion(
        criterion="absolute",  
        tolerable_loss=0.05  # We will tolerate up to 0.05 less accuracy in the quantized model
    ),
    approach="static", 
    device='cpu', 
    quant_level=1,
    quant_format="QOperator", 
    recipes={"graph_optimization_level": "ENABLE_EXTENDED"}, 
    calibration_sampling_size=128
)

# Find the best quantized model meeting the accuracy criterion
q_model = quantization.fit(
    model=fp32_model, 
    conf=config_ptq, 
    calib_dataloader=eval_dataloader,
    eval_dataloader=eval_dataloader, 
    eval_metric=neural_compressor.metric.Metric(name='topk')
)

# Save quantized model
q_model.save_model_to_file("models/food11_quantized_aggressive.onnx")

Download the `food11_quantized_aggressive.onnx` model from inside the `models` directory.

To see the effect of the graph optimizations, we can visualize the models using [Netron](https://netron.app/). Upload the original `food11.onnx` and review the graph. Then, upload the `food11_quantized_aggressive.onnx` and see what has changed in the quantized graph.

Note that within the parameters for each quantized operation, we now have a “scale” and “zero point” - these are used to convert the FP32 values to INT8 values, as described above. The optimal scale and zero point for weights is determined by the fitted weights themselves, but the calibration dataset was required to find the optimal scale and zero point for activations.

Let’s get the size of the quantized model on disk:

In [ ]:
# runs in jupyter container on node-serve-model
onnx_model_path = "models/food11_quantized_aggressive.onnx"
model_size = os.path.getsize(onnx_model_path) 
print(f"Model Size on Disk: {model_size/ (1e6) :.2f} MB")

Next, evaluate the quantized model.

In [ ]:
# runs in jupyter container on node-serve-model
onnx_model_path = "models/food11_quantized_aggressive.onnx"
ort_session = ort.InferenceSession(onnx_model_path, providers=['CPUExecutionProvider'])
benchmark_session(ort_session)

<!-- 

On AMD EPYC

Model Size on Disk: 2.42 MB
Accuracy: 87.12% (2916/3347 correct)
Inference Latency (single sample, median): 7.52 ms
Inference Latency (single sample, 95th percentile): 7.78 ms
Inference Latency (single sample, 99th percentile): 7.84 ms
Inference Throughput (single sample): 132.40 FPS
Batch Throughput: 899.98 FPS

Model Size on Disk: 2.42 MB
Accuracy: 87.12% (2916/3347 correct)
Inference Latency (single sample, median): 7.85 ms
Inference Latency (single sample, 95th percentile): 8.14 ms
Inference Latency (single sample, 99th percentile): 8.26 ms
Inference Throughput (single sample): 126.58 FPS
Batch Throughput: 739.48 FPS

On Intel

Execution provider: ['CPUExecutionProvider']
Accuracy: 89.87% (3008/3347 correct)
Inference Latency (single sample, median): 2.51 ms
Inference Latency (single sample, 95th percentile): 2.60 ms
Inference Latency (single sample, 99th percentile): 2.71 ms
Inference Throughput (single sample): 396.18 FPS
Batch Throughput: 2057.18 FPS


-->

Let’s try a more conservative approach to static quantization next - we’ll allow an accuracy loss only up to **0.01**.

This time, we will see that the quantizer tries a few different “recipes” - in many of them, only some of the operations are quantized, in order to try and reach the target accuracy. After each tuning attempt, it tests the quantized model on the evaluation dataset, to see if it meets the accuracy criterion; if not, it tries again.

In [ ]:
# runs in jupyter container on node-serve-model
# Load ONNX model into Intel Neural Compressor
model_path = "models/food11.onnx"
fp32_model = neural_compressor.model.onnx_model.ONNXModel(model_path)

# Configure the quantizer
config_ptq = neural_compressor.PostTrainingQuantConfig(
    accuracy_criterion = neural_compressor.config.AccuracyCriterion(
        criterion="absolute",  
        tolerable_loss=0.01  # We will tolerate up to 0.01 less accuracy in the quantized model
    ),
    approach="static", 
    device='cpu', 
    quant_level=0,  # 0 is a less aggressive quantization level
    quant_format="QOperator", 
    recipes={"graph_optimization_level": "ENABLE_EXTENDED"}, 
    calibration_sampling_size=128
)

# Find the best quantized model meeting the accuracy criterion
q_model = quantization.fit(
    model=fp32_model, 
    conf=config_ptq, 
    calib_dataloader=eval_dataloader,
    eval_dataloader=eval_dataloader, 
    eval_metric=neural_compressor.metric.Metric(name='topk')
)

# Save quantized model
q_model.save_model_to_file("models/food11_quantized_conservative.onnx")

Download the `food11_quantized_conservative.onnx` model from inside the `models` directory.

To see the effect of the quantization, we can visualize the models using [Netron](https://netron.app/). Upload the `food11_quantized_conservative.onnx` and see what has changed in the quantized graph, relative to the “aggressive quantization” graph.

In this graph, since only some operations are quantized, we have a “Quantize” node before each quantized operation in the graph, and a “Dequantize” node after.

Let’s get the size of the quantized model on disk:

In [ ]:
# runs in jupyter container on node-serve-model
onnx_model_path = "models/food11_quantized_conservative.onnx"
model_size = os.path.getsize(onnx_model_path) 
print(f"Model Size on Disk: {model_size/ (1e6) :.2f} MB")

Next, evaluate the quantized model. While we see some savings in model size relative to the unquantized model, the additional quantize and dequantize operations can make the inference time much slower.

However, these tradeoffs vary from one model to the next, and across implementations and hardware. In some cases, the quantize-dequantize model may still have faster inference times than the unquantized models.

In [ ]:
# runs in jupyter container on node-serve-model
onnx_model_path = "models/food11_quantized_conservative.onnx"
ort_session = ort.InferenceSession(onnx_model_path, providers=['CPUExecutionProvider'])
benchmark_session(ort_session)

<!--

on AMD EPYC

Model Size on Disk: 6.01 MB
Accuracy: 90.20% (3019/3347 correct)
Inference Latency (single sample, median): 10.20 ms
Inference Latency (single sample, 95th percentile): 10.39 ms
Inference Latency (single sample, 99th percentile): 10.66 ms
Inference Throughput (single sample): 97.87 FPS
Batch Throughput: 277.23 FPS

On intel

Execution provider: ['CPUExecutionProvider']
Accuracy: 90.44% (3027/3347 correct)
Inference Latency (single sample, median): 6.60 ms
Inference Latency (single sample, 95th percentile): 6.66 ms
Inference Latency (single sample, 99th percentile): 6.68 ms
Inference Throughput (single sample): 151.36 FPS
Batch Throughput: 540.19 FPS

-->
<!--


::: {.cell .markdown}

### Quantization aware training

To achieve the best of both worlds - high accuracy, but the small model size and faster inference time of a quantized model - we can try quantization aware training. In QAT, the effect of quantization is "simulated" during training, so that we learn weights that are more robust to quantization. Then, when we quantize the model, we can achieve better accuracy.

:::

-->

When you are done, download the fully executed notebook from the Jupyter container environment for later reference. (Note: because it is an executable file, and you are downloading it from a site that is not secured with HTTPS, you may have to explicitly confirm the download in some browsers.)

Also download the models from inside the `models` directory.